### Preprocessing data from AAL90 atlas

#### Import packages

In [ ]:
import os
import scipy.io as sio
import numpy as np
from tqdm import tqdm
from scipy.io import loadmat

#### Set directory

In [50]:
# Get directory with all data (controls and schizophrenic patients)
data_dir = 'D:\\Cogmaster\\M2\\stage\\data\\TS\\AAL\\UCLA'
# Set directory to save processed data
processed_data_dir = 'D:\\Cogmaster\\M2\\stage\\data\\processed_AAL90'

# Get folder names for specific conditions
ctr_folder = 'parceladas_CTRL'
scz_folder = 'parceladas_SCHZ'

#### Set parameters

In [24]:
# Get conditon names 
conds = ['CTR', 'SCZ'] # Conditions: Control and Schizophrenia
n_conds = len(conds)  # Number of conditions
n_rois = 90 # Number of ROIs (Regions of Interest) for AAL90 parcellation

#### Load files

In [25]:
# Get full paths and verify that the number of files in both folders is equal
ctr_data_dir = os.path.join(data_dir, ctr_folder)
scz_data_dir = os.path.join(data_dir, scz_folder)
assert len(os.listdir(ctr_data_dir)) == len(os.listdir(scz_data_dir)), "The number of control and schizophrenia files should be equal."
n_files = len(os.listdir(ctr_data_dir))  # Total number of files for each condition

#### Initialize variable to store data

In [26]:
# Load one file to get number of time points
first_file_name = os.listdir(ctr_data_dir)[0]  # Get the first file name from the control folder
path_first_file = os.path.join(ctr_data_dir, first_file_name)
first_file = loadmat(path_first_file)
first_array = first_file['func_roi']  
n_timepoints = first_array.shape[1]

# Initialize arrays
TS_CTR = np.zeros((n_files, n_timepoints, n_rois))
TS_SCZ = np.zeros((n_files, n_timepoints, n_rois))

#### Keep only files with same amount of time points

In [37]:
# Build sorted file lists once and keep only .mat files
file_names = {
    'CTR': sorted([file_name for file_name in os.listdir(ctr_data_dir) if file_name.endswith('.mat')]),
    'SCZ': sorted([file_name for file_name in os.listdir(scz_data_dir) if file_name.endswith('.mat')]),
}

# Store files that do not match the reference number of time points
bad_files = {'CTR': [], 'SCZ': []}
valid_file_names = {'CTR': [], 'SCZ': []}

for cond, data_dir_cond in [('CTR', ctr_data_dir), ('SCZ', scz_data_dir)]:
    for file_name in file_names[cond]:
        path_onefile = os.path.join(data_dir_cond, file_name)
        n_timepoints_file = loadmat(path_onefile)['func_roi'].shape[1]

        if n_timepoints_file == n_timepoints:
            valid_file_names[cond].append(file_name)
        else:
            bad_files[cond].append(file_name)
            print(
                f"Skipping {file_name} in {cond}: "
                f"{n_timepoints_file} time points (expected {n_timepoints})."
            )

print(f"CTR mismatches: {bad_files['CTR']}")
print(f"SCZ mismatches: {bad_files['SCZ']}")

# Keep both groups aligned in number of files
n_files_clean = min(len(valid_file_names['CTR']), len(valid_file_names['SCZ']))
removed_for_balance = {
    'CTR': valid_file_names['CTR'][n_files_clean:],
    'SCZ': valid_file_names['SCZ'][n_files_clean:],
}
valid_file_names['CTR'] = valid_file_names['CTR'][:n_files_clean]
valid_file_names['SCZ'] = valid_file_names['SCZ'][:n_files_clean]

if removed_for_balance['CTR']:
    print(f"Removed extra CTR files to balance counts: {removed_for_balance['CTR']}")
if removed_for_balance['SCZ']:
    print(f"Removed extra SCZ files to balance counts: {removed_for_balance['SCZ']}")

n_files = n_files_clean

Skipping sub-10524_AAL.mat in CTR: 128 time points (expected 152).
CTR mismatches: ['sub-10524_AAL.mat']
SCZ mismatches: []
Removed extra SCZ files to balance counts: ['sub-50085_AAL.mat']


#### Iterate over files to load all data

In [ ]:
for j in range(n_conds):
    cond = conds[j]
    print(f"Processing condition: {cond}")
    if cond == 'CTR':
        data_dir_cond = ctr_data_dir
        file_names = valid_file_names['CTR']
    else:
        data_dir_cond = scz_data_dir
        file_names = valid_file_names['SCZ']

    # Load all .mat files for the current condition
    for i, file_name in enumerate(tqdm(file_names)):
        if file_name.endswith('.mat'):
            path_onefile = os.path.join(data_dir_cond, file_name)
            file_data = loadmat(path_onefile)
            BOLD_data_AAL160 = file_data['func_roi']  # shape (n_regions=160, time_points)
            BOLD_data_AAL90 = BOLD_data_AAL160[:90, :]  # shape (n_regions=90, time_points)
            if cond == 'CTR':
                TS_CTR[i, :, :] = BOLD_data_AAL90.T
            else:
                TS_SCZ[i, :, :] = BOLD_data_AAL90.T

# Reshape for coherence with original dataset 
TS_CTR = TS_CTR.reshape(n_files, n_rois, n_timepoints)
TS_SCZ = TS_SCZ.reshape(n_files, n_rois, n_timepoints)

print(f"Final shapes: TS_CTR: {TS_CTR.shape}, TS_SCZ: {TS_SCZ.shape}")

Processing condition: CTR


100%|██████████| 49/49 [00:00<00:00, 248.25it/s]


Processing condition: SCZ


100%|██████████| 49/49 [00:00<00:00, 259.82it/s]

Final shapes: TS_CTR: (50, 152, 90), TS_SCZ: (50, 152, 90)


In [ ]:
TS_CTR = np.zeros((n_files, n_timepoints, n_rois))
TS_SCZ = np.zeros((n_files, n_timepoints, n_rois))

#### Save data

In [ ]:
np.save(os.path.join(processed_data_dir, 'TS_Control.npy'), TS_CTR)
np.save(os.path.join(processed_data_dir, 'TS_Shizo.npy'), TS_SCZ)